# Dog Survey Data Cleaning Demo (Refactored Pipeline)

This notebook implements a structured data quality pipeline:

1. Completeness Analysis
2. Consistency Analysis
3. Duplicate Analysis
4. Cleaning Pipeline Execution
5. Before vs After Validation (Statistical + Visual)


In [ ]:
%pip install pandas numpy matplotlib seaborn

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../../'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from src.scripts.profiling import dataset_summary, inconsistency_summary
from src.scripts.completeness import (
    completeness_summary, completeness_report,
    fix_title_completeness,
    fix_email_completeness,
    fix_amount_completeness,
    fix_dog_size_completeness,
    fix_dog_gender_completeness,
    fix_dog_age_completeness
)
from src.scripts.consistency import (
    inconsistency_summary, inconsistency_report,
    fix_title_consistency,
    fix_email_consistency,
    fix_amount_consistency,
    fix_dog_size_consistency,
    fix_dog_gender_consistency,
    fix_dog_age_consistency
)
from src.scripts.duplicates import (
    duplicate_summary,
    duplicate_report,
    remove_duplicates
)
from src.utils.helpers import (
    normalize_missing,
    drop_noise_columns,
    show_step_changes,
    show_duplicate_removal
)

sns.set(style="whitegrid")

## Phase 1 — Load Data

In [ ]:
df = pd.read_csv('../../data/raw/dog_survey.csv', dtype=str, keep_default_na=False)
df.columns = df.columns.str.strip().str.lower()

df_before_drop = df.copy()

df = drop_noise_columns(df)
df = normalize_missing(df)

df_after_drop = df.copy()

df_before = df.copy()

print("\nBefore Drop: ")
display(df_before_drop)
print("\nAfter Drop: ")
display(df_after_drop)

display(dataset_summary(df, 'RAW DATA'))

## Phase 2 — Completeness Analysis

In [ ]:
completeness_report(df, 'RAW DATA')
comp_before = completeness_summary(df)

In [ ]:
# Visualization — Missing Values
plt.figure(figsize=(10,4))
df.isna().sum().sort_values().plot(kind='barh')
plt.title('Missing Values Per Column (Before Cleaning)')
plt.show()

## Phase 3 — Consistency Analysis

In [ ]:
summary_before, mask_before = inconsistency_summary(df)

In [ ]:
inconsistency_report(df, 'RAW DATA')

In [ ]:
# Visualization — Inconsistencies
plt.figure(figsize=(8,4))
summary_before['Issue Count'].plot(kind='bar')
plt.title('Inconsistencies by Field (Before Cleaning)')
plt.xticks(rotation=45)
plt.show()

## Phase 4 — Duplicate Analysis

In [ ]:
dup_summary_before, dup_rows_before, dup_groups_before = duplicate_summary(df)
duplicate_report(df, 'RAW DATA')

In [ ]:
# Visualization — Duplicates
dup_mask = df_before.duplicated(keep=False)

dup_rows = dup_mask.sum()
total_rows = len(df_before)

unique_rows = total_rows - dup_rows

plt.figure(figsize=(5,5))
plt.pie(
    [unique_rows, dup_rows],
    labels=["Unique Rows", "Duplicated Rows (All Occurrences)"],
    autopct="%1.1f%%"
)

plt.title("Dataset Composition: Duplicate Impact (Before Cleaning)")
plt.show()

## Phase 5 — Cleaning Pipeline

In [ ]:
total_changes = {}

# Remove duplicates
before = df.copy()
df, _ = remove_duplicates(df)
total_changes['duplicates_removed'] = show_duplicate_removal(before, df)

In [ ]:
# Title
before = df.copy()
df = fix_title_completeness(df)
after_completeness = df.copy()
total_changes["title"] = show_step_changes(
    "Title Completeness Cleaning",
    before,
    after_completeness,
    focus_cols=["title"]
)
df = fix_title_consistency(df)
after_consistency = df.copy()
total_changes["title"] = show_step_changes(
    "Title Consistency Cleaning",
    after_completeness,
    after_consistency,
    focus_cols=["title"]
)

In [ ]:
# Email
before = df.copy()
df = fix_email_completeness(df)
after_completeness = df.copy()
total_changes["email"] = show_step_changes(
    "Email Completeness Cleaning", before, after_completeness, focus_cols=["email"]
)
df = fix_email_consistency(df)
after_consistency = df.copy()
total_changes["email"] = show_step_changes(
    "Email Consistency Cleaning", after_completeness, after_consistency, focus_cols=["email"]
)

In [ ]:
# Amount
before = df.copy()
df = fix_amount_completeness(df)
after_completeness = df.copy()
total_changes["amount"] = show_step_changes(
    "Amount Completeness Cleaning", before, after_completeness, focus_cols=["amount_spent_on_dog_food"]
)
df = fix_amount_consistency(df)
after_consistency = df.copy()
total_changes["amount"] = show_step_changes(
    "Amount Consistency Cleaning", after_completeness, after_consistency, focus_cols=["amount_spent_on_dog_food"]
)

In [ ]:
# Dog size
before = df.copy()
df = fix_dog_size_completeness(df)
after_completeness = df.copy()
total_changes["dog_size"] = show_step_changes(
    "Dog Size Completeness Cleaning", before, after_completeness, focus_cols=["dog_size"]
)
df = fix_dog_size_consistency(df)
after_consistency = df.copy()
total_changes["dog_size"] = show_step_changes(
    "Dog Size Consistency Cleaning", after_completeness, after_consistency, focus_cols=["dog_size"]
)

In [ ]:
# Dog gender
before = df.copy()
df = fix_dog_gender_completeness(df)
after_completeness = df.copy()
total_changes["dog_gender"] = show_step_changes(
    "Dog Gender Completeness Cleaning", before, after_completeness, focus_cols=["dog_gender"]
)
df = fix_dog_gender_consistency(df)
after_consistency = df.copy()
total_changes["dog_gender"] = show_step_changes(
    "Dog Gender Consistency Cleaning", after_completeness, after_consistency, focus_cols=["dog_gender"]
)

In [ ]:
# Dog age
before = df.copy()
df = fix_dog_age_completeness(df)
after_completeness = df.copy()
total_changes["dog_age"] = show_step_changes(
    "Dog Age Completeness Cleaning", before, after_completeness, focus_cols=["dog_age"]
)
df = fix_dog_age_consistency(df)
after_consistency = df.copy()
total_changes["dog_age"] = show_step_changes(
    "Dog Age Consistency Cleaning", after_completeness, after_consistency, focus_cols=["dog_age"]
)

## Phase 6 — Before vs After Validation

In [ ]:
display(dataset_summary(df_before, "BEFORE"))
display(dataset_summary(df, "AFTER"))

In [ ]:
# Completeness comparison
comp_after = completeness_summary(df)

comp_compare = pd.DataFrame({
    'Before': comp_before['Missing Count'],
    'After': comp_after['Missing Count']
}).fillna(0)

comp_compare.plot(kind='bar', figsize=(10,4))
plt.title('Completeness Comparison')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Consistency comparison
summary_after, mask_after = inconsistency_summary(df)

compare = pd.DataFrame({
    'Before': summary_before['Issue Count'],
    'After': summary_after['Issue Count']
}).fillna(0)

compare.plot(kind='bar', figsize=(10,4))
plt.title('Consistency Reduction')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Duplicate comparison
dup_mask_after = df.duplicated(keep=False)

dup_rows_after = dup_mask_after.sum()
total_rows_after = len(df)

unique_rows_after = total_rows_after - dup_rows_after

plt.figure(figsize=(5,5))
plt.pie(
    [unique_rows_after, dup_rows_after],
    labels=["Unique Rows", "Duplicated Rows (All Occurrences)"],
    autopct="%1.1f%%"
)

plt.title("Dataset Composition: Duplicate Impact (After Cleaning)")
plt.show()

## Phase 7 — Cleaning Impact

In [ ]:
impact_df = pd.DataFrame(list(total_changes.items()), columns=['Step', 'Changes'])
impact_df.sort_values('Changes', ascending=False, inplace=True)
impact_df

In [ ]:
impact_df.plot(kind='bar', figsize=(10,4))
plt.title('Cleaning Impact Per Step')
plt.xticks(rotation=45)
plt.show()

## Export

In [ ]:
df.to_csv('../../data/processed/dog_survey_cleaned.csv', index=False)
print('Export completed')